In [9]:
import requests

#Seleccione el recurso descargar
url = "https://en.wikipedia.org/wiki/List_of_Spotify_streaming_records"

response = requests.get(url, headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36"})

response.raise_for_status()
html = response.text


In [10]:
response

<Response [200]>

In [11]:
#Paso 3: Transformar el HTML
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# Buscar tablas en la página
tables = soup.find_all("table", class_="wikitable")

print(f"Número de tablas encontradas: {len(tables)}")


# Leer todas las tablas del HTML
df_list = pd.read_html(html)

# Seleccionamos la primera tabla relevante
df = df_list[0]

df.head()

ModuleNotFoundError: No module named 'bs4'

In [12]:
#Paso 4: Procesar y limpiar el DataFrame Revisar columnas
df.columns

NameError: name 'df' is not defined

In [ ]:
#Limpiar columnas numéricas (eliminar $ y B)


# Ejemplo: limpiar todas las columnas tipo object
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace("B", "", regex=False)
            .str.replace("M", "", regex=False)
        )

In [ ]:
#Eliminar filas vacías o sin información
df = df.dropna()
df = df.reset_index(drop=True)

df.head()

In [ ]:
#5: Almacenar los datos en SQLite

import sqlite3

conn = sqlite3.connect("spotify_records.db")
cursor = conn.cursor()

# Crear tabla (estructura genérica)
cursor.execute("""
CREATE TABLE IF NOT EXISTS spotify_records (
    id INTEGER PRIMARY KEY AUTOINCREMENT
)
""")

In [ ]:
#Ajustar la tabla a las columnas reales del DataFrame

Creamos la tabla dinámicamente:

# Crear columnas dinámicamente
columns_sql = ", ".join([f'"{col}" TEXT' for col in df.columns])

cursor.execute(f"""
CREATE TABLE IF NOT EXISTS spotify_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    {columns_sql}
)
""")

In [ ]:
#Insertar los datos
placeholders = ", ".join(["?"] * len(df.columns))
columns = ", ".join(df.columns)

for _, row in df.iterrows():
    cursor.execute(
        f"INSERT INTO spotify_data ({columns}) VALUES ({placeholders})",
        tuple(row)
    )

conn.commit()
conn.close()




In [ ]:
#Paso 6: Visualización de los datos
import matplotlib.pyplot as plt